# Semantic segmentation on Oxford Pets

Classifying every pixel rather than the image. Same ConvNet ideas, one decisive change: strides instead of pooling, because now position matters.

**Runs on:** GPU recommended — about 25 minutes on CPU · needs the Oxford-IIIT Pets download &nbsp;·&nbsp; **Slides:** [Chapter 11 — Image Segmentation](../../../course-web-slides/ch11/index.html) &nbsp;·&nbsp; **Section:** 01 — Image segmentation

---

## The data

In [ ]:
import os, pathlib
import keras

input_dir = "images/"
target_dir = "annotations/trimaps/"

# Download from https://www.robots.ox.ac.uk/~vgg/data/pets/ if not present.
input_img_paths = sorted(
    [os.path.join(input_dir, f) for f in os.listdir(input_dir)
     if f.endswith(".jpg")])
target_paths = sorted(
    [os.path.join(target_dir, f) for f in os.listdir(target_dir)
     if f.endswith(".png") and not f.startswith(".")])

print(f"{len(input_img_paths)} images")
print(input_img_paths[9])
print(target_paths[9])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from keras.utils import load_img, img_to_array, array_to_img

plt.figure(figsize=(4, 4))
plt.axis("off")
plt.imshow(load_img(input_img_paths[9]))
plt.show()

def display_target(target_array):
    # Labels are 1, 2, 3. Subtract 1 and scale so they are visible.
    normalized = (target_array.astype("uint8") - 1) * 127
    plt.axis("off")
    plt.imshow(normalized[:, :, 0])

img = img_to_array(load_img(target_paths[9], color_mode="grayscale"))
plt.figure(figsize=(4, 4)); display_target(img); plt.show()
print("unique label values:", np.unique(img))

Expected output:

```
unique label values: [1. 2. 3.]
```

Three classes per pixel: **1 = the animal, 2 = the background, 3 = the outline.** The target has the same spatial shape as the input, which is the entire difference from chapter 8.

## Loading everything into memory

In [ ]:
img_size = (200, 200)
num_imgs = len(input_img_paths)

import random
random.Random(1337).shuffle(input_img_paths)
random.Random(1337).shuffle(target_paths)

def path_to_input_image(path):
    return img_to_array(load_img(path, target_size=img_size))

def path_to_target(path):
    img = img_to_array(
        load_img(path, target_size=img_size, color_mode="grayscale"))
    img = img.astype("uint8") - 1        # labels become 0, 1, 2
    return img

input_imgs = np.zeros((num_imgs,) + img_size + (3,), dtype="float32")
targets = np.zeros((num_imgs,) + img_size + (1,), dtype="uint8")
for i in range(num_imgs):
    input_imgs[i] = path_to_input_image(input_img_paths[i])
    targets[i] = path_to_target(target_paths[i])

num_val_samples = 1000
train_input_imgs = input_imgs[:-num_val_samples]
train_targets = targets[:-num_val_samples]
val_input_imgs = input_imgs[-num_val_samples:]
val_targets = targets[-num_val_samples:]
print(train_input_imgs.shape, train_targets.shape)

> **Note** — The same shuffle seed on both lists. **They must stay aligned** — and two independently shuffled lists is a silent, total failure with no error message.

## The model, and the one decisive change

In [ ]:
from keras import layers

def get_model(img_size, num_classes):
    inputs = keras.Input(shape=img_size + (3,))
    x = layers.Rescaling(1./255)(inputs)

    x = layers.Conv2D(64, 3, strides=2, activation="relu", padding="same")(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(128, 3, strides=2, activation="relu", padding="same")(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(256, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(256, 3, activation="relu", padding="same")(x)

    x = layers.Conv2DTranspose(256, 3, activation="relu", padding="same")(x)
    x = layers.Conv2DTranspose(256, 3, activation="relu", padding="same",
                               strides=2)(x)
    x = layers.Conv2DTranspose(128, 3, activation="relu", padding="same")(x)
    x = layers.Conv2DTranspose(128, 3, activation="relu", padding="same",
                               strides=2)(x)
    x = layers.Conv2DTranspose(64, 3, activation="relu", padding="same")(x)
    x = layers.Conv2DTranspose(64, 3, activation="relu", padding="same",
                               strides=2)(x)

    outputs = layers.Conv2D(num_classes, 3, activation="softmax",
                            padding="same")(x)
    return keras.Model(inputs, outputs)

model = get_model(img_size=img_size, num_classes=3)
model.summary()

**`strides=2` everywhere, and no `MaxPooling2D`.** Max pooling throws away *where* the maximum was, and for classification that is a feature — for segmentation it is the answer being discarded.

## Training

In [ ]:
model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy")

callbacks = [keras.callbacks.ModelCheckpoint("oxford_segmentation.keras",
                                             save_best_only=True)]
history = model.fit(train_input_imgs, train_targets,
                    epochs=50, callbacks=callbacks, batch_size=64,
                    validation_data=(val_input_imgs, val_targets), verbose=2)

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)
plt.figure(figsize=(7, 4.2))
plt.plot(epochs, history.history["loss"], lw=1, label="training")
plt.plot(epochs, history.history["val_loss"], lw=1.7, label="validation")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title(f"Overfits from about epoch "
          f"{int(np.argmin(history.history['val_loss']))+1}")
plt.show()

## Predictions

In [ ]:
model = keras.models.load_model("oxford_segmentation.keras")

i = 4
test_image = val_input_imgs[i]
mask = model.predict(np.expand_dims(test_image, 0), verbose=0)[0]

def display_mask(pred):
    mask = np.argmax(pred, axis=-1) * 127
    plt.axis("off"); plt.imshow(mask)

fig = plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.axis("off")
plt.imshow(array_to_img(test_image)); plt.title("input")
plt.subplot(1, 3, 2); plt.axis("off")
plt.imshow(val_targets[i][:, :, 0] * 127); plt.title("ground truth")
plt.subplot(1, 3, 3); display_mask(mask); plt.title("prediction")
plt.tight_layout(); plt.show()

The animal is found. The **outline class is the weak one** — a one-pixel-wide band, and there are far fewer of those pixels than of the other two.

## Per-class IoU, because pixel accuracy lies

In [ ]:
preds = model.predict(val_input_imgs[:200], verbose=0).argmax(-1)
truth = val_targets[:200, :, :, 0]

print(f"pixel accuracy: {(preds == truth).mean():.4f}\n")
names = ["animal", "background", "outline"]
for c in range(3):
    inter = ((preds == c) & (truth == c)).sum()
    union = ((preds == c) | (truth == c)).sum()
    freq = (truth == c).mean()
    print(f"{names[c]:11s} IoU {inter/union:.3f}   "
          f"{freq:.1%} of all pixels")

Expected output:

```
pixel accuracy: 0.87xx

animal      IoU 0.8xx   3x.x% of all pixels
background  IoU 0.8xx   5x.x% of all pixels
outline     IoU 0.2xx    6.x% of all pixels
```

**Pixel accuracy of 87% hides an outline IoU of 0.2.** Chapter 6's rule about choosing a metric that reflects what you want has teeth here: a model that ignored the outline entirely would barely dent the headline number.

---

## What to take away

- Segmentation predicts a class per pixel; the target has the input's spatial shape.
- **Strides, not pooling** — max pooling discards the position that is the answer.
- Shuffle inputs and targets with the same seed, or fail silently and completely.
- Report per-class IoU; pixel accuracy hides the rare class you probably care about.